# SFS / SHS survey moments

Load Canadian Survey of Financial Security (1999–2023) and Survey of Household Spending (Tier A 2017/19/23 + Tier B 1997–2009), then compute **weighted** aggregates.

Requires `data/local_paths.yaml` (see `local_paths.example.yaml`). Microdata stays outside git.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.survey import (
    load_local_paths,
    load_sfs,
    load_shs_all,
    sfs_aggregate_moments,
    shs_cy_moments,
)

paths = load_local_paths()
for k, v in paths.items():
    print(f"{k}: {v}  exists={v.exists()}")

## SFS: ownership, housing/financial shares of NW, NW/Y

Primary net worth excludes autos:

$$NW = H^{gross}-D^H + F^{gross}-D^F = H^{net}+F^{net}.$$

Shares are aggregate ratios $\sum w X / \sum w\, NW$ (gross and net).

In [ ]:
sfs = load_sfs()
print("years", sorted(sfs["year"].dropna().unique().tolist()), "n=", len(sfs))
# Identity check
err = (sfs["nw"] - (sfs["housing_net"] + sfs["financial_net"])).abs().max()
print("max |nw - (H_net+F_net)|", float(err))

sfs_m = sfs_aggregate_moments(sfs, by="year")
display(sfs_m)

out = ROOT / "output" / "survey"
out.mkdir(parents=True, exist_ok=True)
sfs_m.to_csv(out / "sfs_aggregate_moments.csv", index=False)

## SHS: aggregate consumption / after-tax income

Tier A uses `TC001 / (HHTOTINC - TX010)`; Tier B uses consolidated `consumption / net_income`.
Do not splice A and B into one trend.

In [ ]:
shs = load_shs_all()
print(shs.groupby(["tier", "year"]).size())
shs_m = shs_cy_moments(shs, by=["tier", "year"])
display(shs_m)
shs_m.to_csv(out / "shs_cy_moments.csv", index=False)